# MA4703 — Etapa E: control óptimo y principio de Pontryagin

Control óptimo **aproximado** para sincronización y desincronización de redes FitzHugh–Nagumo. Se reutilizan todas las etapas A–D. La optimización es directa y sus candidatos se comprueban con ecuaciones adjuntas continuas.

**Advertencia:** satisfacer las ecuaciones necesarias de PMP no demuestra optimalidad global. No se presupone que un control propuesto pueda conseguir un objetivo terminal. Reinicia el kernel antes de ejecutar.

In [ ]:
using Pkg
project = dirname(Base.active_project())
include(joinpath(project, "src", "FitzHughNagumoControl.jl"))
using .FitzHughNagumoControl
using JLD2, LinearAlgebra
println("Proyecto: ",project)

## 1. Parámetros y condiciones iniciales

Usamos N=4 para que la optimización sea manejable. Siempre conservamos el signo original de acoplamiento: `lambda < 0` es atractivo. Si existe el archivo JLD2 de la Etapa B, reutilizamos estados de su ciclo periódico refinado.

In [ ]:
p = FHNParameters(a=0.7,b=0.5,tau=12.5,I=1.4)
N = 4
center = equilibrium(p)
bfile = joinpath(project,"data","processed","etapa_B_resultados.jld2")
if isfile(bfile)
    db=JLD2.load(bfile)
    q=db["parameters"]
    compatible=all(isapprox(getproperty(p,f),getproperty(q,f);atol=1e-12)
                   for f in (:a,:b,:tau,:I))
else
    compatible=false
end
if compatible
    pts=db["orbit_states"]
    x0=zeros(2N)
    for i in 1:N
        j=1+round(Int,(i-1)*(size(pts,2)-1)/N)
        x0[2i-1:2i] .= pts[:,j]
    end
    xsync=synchronized_state(N,pts[:,1])
else
    x0=distributed_initial_state(N,p)
    x0[1:2:end] .+= 0.3
    xsync=synchronized_state(N,[-1.4,-0.2])
end
println("E(x0)=", synchronization_error(x0),"; fases iniciales definidas=",isfinite(phase_order_squared(x0,center)))

## 2. Sincronización — tiempo final libre

La función objetivo es

$$J_{\mathrm{sin}}=\int_0^T\big[\rho+\alpha E(x)+\beta(1-r(x)^2)^2+\gamma(\lambda-\lambda_{ref})^2\big]dt,$$

con $E(x(T))\leq\delta$. Resolvemos un problema de dimensión finita mediante controles constantes por tramos. **Exigimos** la restricción terminal (no sólo una penalización) y la verificamos de nuevo con Tsit5. Si el resultado no es factible, se detiene el experimento de sincronización.

In [ ]:
specS=SyncOCPSpec(p,N,x0;delta=0.03,
        lambda_bounds=(-2.5,0.5),lambda_ref=-0.15,
        rho=1.0,alpha=0.1,beta=0.2,gamma=0.5)
rs=optimize_sync_direct(specS;M=10,substeps=10,T0=30.0,
        T_bounds=(5.0,100.0),initial_lambda=-1.35,
        outer_iterations=4,inner_iterations=35)
println("J discreto=",rs.J,"; T=",rs.T,"; E(T)-delta=",rs.constraint_residual,"; factible=",rs.feasible,"; estado=",rs.solver_status)
println("E(T) verificado con Tsit5=",rs.terminal_E_independent)
rs.feasible || error("No se encontró control factible: NO interpretar este candidato como solución de sincronización.")
println("Control λ=",round.(rs.controls,digits=3))

In [ ]:
simS=simulate_optimal_sync(specS,rs;saveat=0.15)
costS=evaluate_sync_cost(simS.sol,simS.network;
       delta=specS.delta,rho=specS.rho,alpha=specS.alpha,beta=specS.beta,
       gamma=specS.gamma,lambda_ref=specS.lambda_ref,center=specS.center)
ET=synchronization_error(simS.sol.u[end])
println("E(T) independiente=",ET)
ET<=specS.delta || error("E(T)>δ: no mostrar ni guardar este candidato como factible.")
println("Primera llegada=",costS.status,"; t=",costS.arrival_time,"; J=",costS.arrival_J)
plot_optimal_sync(rs,specS,simS,costS)

## 3. Adjuntas y residuos de Pontryagin — sincronización

En el caso normal regular se requiere $p(T)=\mu\nabla E(x(T))$ y $H(T)=0$. Una diferencia importante entre el control candidato y la ley proyectada implica que las condiciones necesarias **todavía no están satisfechas**.

In [ ]:
diagS=pmp_diagnostics_sync(specS,rs,simS.sol;samples=80)
println("max |lambda - lambda_PMP| = ",diagS.max_projected_control_residual)
println("H(T) = ",diagS.terminal_H,"; mu≈",diagS.mu)
plot_pmp_sync(diagS)

## 4. Desincronización — corrientes individuales

Desde estados exactamente iguales y con $\sum_i u_i=0$, resolvemos

$$J_{\mathrm{des}}=\int_0^T\left[\alpha_d r^2+\gamma_d\sum_i u_i^2\right]dt+\kappa r^2(x(T)).$$

El control nulo es un extremal trivial. Por ello se usa multistart y se informa el costo sin control con los mismos pesos. Los pesos aquí son ilustrativos y distintos de los usados en la Etapa D.

In [ ]:
specD=DesyncOCPSpec(p,N,xsync;T=25.0,lambda0=-0.15,
         Umax=0.8,alpha_d=3.0,gamma_d=0.05,kappa=8.0)
rd=optimize_desync_direct(specD;M=10,substeps=9,maxiter=35,seed_amplitude=0.6)
println("J candidato=",rd.J,"; J sin control=",rd.baseline_J)
println("r²(T)=",rd.terminal_r2,"; max suma de corrientes=",rd.max_current_sum)
simD=simulate_optimal_desync(specD,rd;saveat=0.15)
costD=evaluate_desync_cost(simD.sol,simD.network;
        alpha_d=specD.alpha_d,gamma_d=specD.gamma_d,
        kappa=specD.kappa,Umax=specD.Umax,center=specD.center)
println("J recalculado con Tsit5=",costD.total_J)
plot_optimal_desync(rd,specD,simD,costD)

## 5. Adjuntas y condiciones necesarias para las corrientes

Se calcula hacia atrás $p(T)=\kappa\nabla r^2(x(T))$ y se compara el control aplicado con el minimizador proyectado que mantiene simultáneamente $|u_i|\leq U_{max}$ y $\sum_i u_i=0$.

In [ ]:
diagD=pmp_diagnostics_desync(specD,rd,simD.sol;samples=80)
println("max |u - u_PMP|=",diagD.max_projected_control_residual)
println("max suma de corrientes=",diagD.max_current_sum)
plot_pmp_desync(diagD)

## 6. Guardar datos (sin volver a integrar)

Los archivos contienen controles, series temporales, estados y residuos; no se serializan objetos ODE ni funciones. Antes de pasar a HJB, revisar factibilidad, continuidad numérica y residuos de PMP.

In [ ]:
datadir=joinpath(project,"data","processed")
mkpath(datadir)
save_optimal_result(joinpath(datadir,"etapa_E_sync_notebook.jld2"),specS,rs,simS,diagS)
save_optimal_result(joinpath(datadir,"etapa_E_desync_notebook.jld2"),specD,rd,simD,diagD)
println("Datos guardados en ",datadir)